In [2]:
from pathlib import Path
import json

import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
TEST_SIZE = 0.2

DATA_PATH = Path("router_merged_3domains.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Could not find training/router/router_merged_3domains.csv from the current working directory.")

OUTPUT_DIR = DATA_PATH.parent / "weights"
MODEL_PATH = OUTPUT_DIR / "router_domain_classifier.joblib"
METADATA_PATH = OUTPUT_DIR / "router_domain_classifier_metadata.json"

In [3]:
df = pd.read_csv(DATA_PATH)

required_columns = {"domain", "title", "category"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

df = df[["domain", "title", "category"]].dropna(subset=["domain"]).copy()
df["title"] = df["title"].fillna("").astype(str)
df["category"] = df["category"].fillna("").astype(str)
df["domain"] = df["domain"].astype(str)

print("Data path:", DATA_PATH)
print("Shape:", df.shape)
display(df["domain"].value_counts().rename("count"))
display(df.head())

Data path: router_merged_3domains.csv
Shape: (4357, 3)


domain
hotel         2016
restaurant    1793
hospital       548
Name: count, dtype: int64

,domain,title,category
0,hospital,Trung Tâm Y Tế Quận Hà Đông,Trung tâm y tế
1,hospital,Trung tâm Y tế quận Hai Bà Trưng,Trung tâm y tế
2,hospital,Trung Tâm Y Tế Quận Hoàn Kiếm,Trung tâm y tế
3,hospital,Bệnh Viện Đa Khoa Hồ Tràm,Trung tâm y tế
4,hospital,Trung Tâm Y Tế Quận Long Biên,Trung tâm y tế


In [4]:
X = df[["title", "category"]]
y = df["domain"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 3485
Test size: 872


In [5]:
def build_pipeline(
    title_ngram_range=(1, 2),
    title_min_df=2,
    category_ngram_range=(1, 2),
    C=1.0,
    class_weight="balanced",
):
    features = ColumnTransformer(
        transformers=[
            (
                "title_tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    ngram_range=title_ngram_range,
                    max_features=10000,
                    min_df=title_min_df,
                    sublinear_tf=True,
                ),
                "title",
            ),
            (
                "category_tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    ngram_range=category_ngram_range,
                    min_df=1,
                    sublinear_tf=True,
                ),
                "category",
            ),
        ],
        remainder="drop",
    )

    return Pipeline(
        steps=[
            ("features", features),
            (
                "clf",
                LinearSVC(
                    C=C,
                    class_weight=class_weight,
                    max_iter=5000,
                    dual="auto",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
baseline_pipeline = build_pipeline()

In [6]:
baseline_scores = cross_validate(
    baseline_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=["accuracy", "f1_macro", "f1_weighted"],
    n_jobs=-1,
)

baseline_summary = pd.DataFrame(
    {
        "metric": ["accuracy", "f1_macro", "f1_weighted"],
        "mean": [
            baseline_scores["test_accuracy"].mean(),
            baseline_scores["test_f1_macro"].mean(),
            baseline_scores["test_f1_weighted"].mean(),
        ],
        "std": [
            baseline_scores["test_accuracy"].std(),
            baseline_scores["test_f1_macro"].std(),
            baseline_scores["test_f1_weighted"].std(),
        ],
    }
)

display(baseline_summary)

,metric,mean,std
0,accuracy,0.997991,0.002661
1,f1_macro,0.997645,0.003645
2,f1_weighted,0.997987,0.002669


In [7]:
param_grid = {
    "features__title_tfidf__ngram_range": [(1, 1), (1, 2)],
    "features__title_tfidf__min_df": [1, 2],
    "features__category_tfidf__ngram_range": [(1, 1), (1, 2)],
    "clf__C": [0.3, 1.0, 3.0],
    "clf__class_weight": ["balanced", None],
}

grid_search = GridSearchCV(
    estimator=baseline_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print("Best CV macro-F1:", grid_search.best_score_)
print("Best params:")
display(grid_search.best_params_)

Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best CV macro-F1: 0.9976453303503645
Best params:


{'clf__C': 0.3,
 'clf__class_weight': 'balanced',
 'features__category_tfidf__ngram_range': (1, 2),
 'features__title_tfidf__min_df': 1,
 'features__title_tfidf__ngram_range': (1, 1)}

In [8]:
grid_results = pd.DataFrame(grid_search.cv_results_)
grid_results = grid_results.sort_values("rank_test_score")

columns_to_show = [
    "rank_test_score",
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "param_features__title_tfidf__ngram_range",
    "param_features__title_tfidf__min_df",
    "param_features__category_tfidf__ngram_range",
    "param_clf__C",
    "param_clf__class_weight",
]

display(grid_results[columns_to_show].head(10))

,rank_test_score,mean_test_score,std_test_score,mean_train_score,param_features__title_tfidf__ngram_range,param_features__title_tfidf__min_df,param_features__category_tfidf__ngram_range,param_clf__C,param_clf__class_weight
6,1,0.997645,0.003645,0.999341,"(1, 1)",2,"(1, 2)",0.3,balanced
4,1,0.997645,0.003645,0.999341,"(1, 1)",1,"(1, 2)",0.3,balanced
15,1,0.997645,0.003645,0.999341,"(1, 2)",2,"(1, 2)",0.3,NaN
14,1,0.997645,0.003645,0.999341,"(1, 1)",2,"(1, 2)",0.3,NaN
12,1,0.997645,0.003645,0.999341,"(1, 1)",1,"(1, 2)",0.3,NaN
31,1,0.997645,0.003645,0.999451,"(1, 2)",2,"(1, 2)",1.0,NaN
23,1,0.997645,0.003645,0.999341,"(1, 2)",2,"(1, 2)",1.0,balanced
22,1,0.997645,0.003645,0.999341,"(1, 1)",2,"(1, 2)",1.0,balanced
30,1,0.997645,0.003645,0.999341,"(1, 1)",2,"(1, 2)",1.0,NaN
46,1,0.997645,0.003645,0.999341,"(1, 1)",2,"(1, 2)",3.0,NaN


In [9]:
best_model = grid_search.best_estimator_
test_pred = best_model.predict(X_test)

print(classification_report(y_test, test_pred, digits=4))

labels = list(best_model.named_steps["clf"].classes_)
confusion = pd.DataFrame(
    confusion_matrix(y_test, test_pred, labels=labels),
    index=[f"actual_{label}" for label in labels],
    columns=[f"pred_{label}" for label in labels],
)
display(confusion)

              precision    recall  f1-score   support

    hospital     0.9910    1.0000    0.9955       110
       hotel     1.0000    0.9950    0.9975       403
  restaurant     0.9972    1.0000    0.9986       359

    accuracy                         0.9977       872
   macro avg     0.9961    0.9983    0.9972       872
weighted avg     0.9977    0.9977    0.9977       872



,pred_hospital,pred_hotel,pred_restaurant
actual_hospital,110,0,0
actual_hotel,1,401,1
actual_restaurant,0,0,359


In [10]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, MODEL_PATH)

metadata = {
    "data_path": str(DATA_PATH),
    "model_path": str(MODEL_PATH),
    "features": ["title", "category"],
    "target": "domain",
    "labels": labels,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "cv": "StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
    "scoring": "f1_macro",
    "best_cv_macro_f1": float(grid_search.best_score_),
    "best_params": grid_search.best_params_,
}

with METADATA_PATH.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Saved model to:", MODEL_PATH)
print("Saved metadata to:", METADATA_PATH)

Saved model to: weights\router_domain_classifier.joblib
Saved metadata to: weights\router_domain_classifier_metadata.json
